# CaFE-Rec: Training on Google Colab (T4 GPU)

This notebook lets you train the CaFE-Rec model on a free Colab T4 GPU.

**Before running this notebook:**
1. Upload the `cafe_rec/` folder to your Google Drive
2. Upload the `cafe_rec/data/processed/` folder (train.pkl, val.pkl, test.pkl, aspect_vocab.json, mappings.json)
3. Set runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU)

## Step 1: Check GPU is available

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found! Go to Runtime -> Change runtime type -> T4 GPU")

## Step 2: Mount Google Drive

Your project folder should be in Google Drive. Update the path below if needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# UPDATE THIS PATH to match where you put cafe_rec/ in Google Drive
PROJECT_DIR = "/content/drive/MyDrive/Recommender_Project/cafe_rec"

import os
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## Step 3: Install dependencies

In [ ]:
!pip install -q torch transformers torch-geometric datasets wandb \
    numpy pandas scikit-learn tqdm sacrebleu rouge-score bert-score \
    sentencepiece accelerate spacy
!python -m spacy download en_core_web_sm -q
print("All dependencies installed!")

## Step 4: Verify data is in place

In [ ]:
required_files = [
    "data/processed/train.pkl",
    "data/processed/val.pkl",
    "data/processed/test.pkl",
    "data/processed/aspect_vocab.json",
    "data/processed/mappings.json",
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    size = f"{os.path.getsize(f) / 1e6:.1f} MB" if exists else "MISSING"
    status = "OK" if exists else "MISSING"
    print(f"  {status:7s} {size:>10s}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll data files found! Ready to train.")
else:
    print("\nSome files are missing. Upload them to Google Drive first.")

## Step 5: Quick smoke test (optional)

Run this to verify everything works before starting the long training.

In [ ]:
!python train.py --dataset amazon_movies --processed_dir data/processed --no_wandb --debug

## Step 6: Train the model

Choose your training size:

| Sample size | Estimated time (T4) | Best for |
|-------------|--------------------|-----------|
| 100,000     | 1-2 hours          | Quick experiments |
| 500,000     | 5-8 hours          | Good results |
| Full (6M)   | 8-15 hours         | Best results (may need Colab Pro) |

**Tip:** Start with 100k to make sure everything works, then scale up.

In [ ]:
# OPTION A: Train on 100k samples (recommended first run, ~1-2 hrs)
!python train.py \
    --dataset amazon_movies \
    --processed_dir data/processed \
    --no_wandb \
    --sample_size 100000

In [ ]:
# OPTION B: Train on full dataset (best quality, ~8-15 hrs)
# Uncomment below and comment out Option A above

# !python train.py \
#     --dataset amazon_movies \
#     --processed_dir data/processed \
#     --no_wandb

## Step 7: Evaluate the trained model

After training finishes, check the results. The best checkpoint is auto-saved.

In [ ]:
# List saved checkpoints
import glob
ckpts = sorted(glob.glob("checkpoints/*.pt"))
for c in ckpts:
    size = os.path.getsize(c) / 1e6
    print(f"  {size:>8.1f} MB  {c}")

if ckpts:
    print(f"\nBest checkpoint: {ckpts[-1]}")

In [ ]:
# Run evaluation on the best checkpoint
!python evaluate.py --help

## Step 8: Download results

Checkpoints are saved in `checkpoints/` on your Google Drive, so they persist after Colab disconnects.

In [ ]:
# Copy checkpoints to a safe location on Drive
!cp -r checkpoints/ /content/drive/MyDrive/Recommender_Project/checkpoints_backup/
print("Checkpoints backed up to Google Drive!")